In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 📚 第2周-Day5：推理优化——KV Cache 与 Flash Attention

> **大模型很强，但也很慢。一个 70B 的模型生成一个 token 可能需要几百毫秒，生成 1000 个 token 就要等几十秒。今天我们学习让大模型"跑得快"的三大核心技术：KV Cache（以空间换时间）、Flash Attention（减少内存访问）和 GQA（平衡效率与质量）。这些技术是大模型从实验室走向生产环境的关键。**


## 📅 学习进度


In [ ]:
W1 ████████████████████ ✓ 已完成（Transformer 基础架构）
W2 ████████████░░░░░░░░ ← 你在这里（Day 5/7）
W3 ░░░░░░░░░░░░░░░░░░░░ 预训练与数据工程
W4 ░░░░░░░░░░░░░░░░░░░░ 微调与对齐
...
W13 ░░░░░░░░░░░░░░░░░░░░ 综合项目


---


## 一、为什么需要推理优化？

### 大模型的"慢"有多严重？

考虑一个 7B 参数的模型：
- 生成 1 个 token 需要做约 14 GFLOPS 运算
- 生成 1000 个 token（约 500 字中文）→ 14 TFLOPS
- 在 RTX 4060Ti 上（约 15 TFLOPS 算力）→ 理论需要 ~1 秒
- 实际上？可能需要 10-30 秒！

为什么理论和实际差距这么大？因为**内存带宽才是瓶颈，不是算力**！

**打个比方**：你的厨师（GPU 算力）刀工极快，但食材（数据）要从很远的仓库（显存）运过来。厨师大部分时间不是在切菜，而是在等食材送达。

推理优化的目标就是减少"等食材"的时间。

### 三大核心技术概览

| 技术 | 解决什么问题 | 核心思想 | 比喻 |
|------|------------|---------|------|
| **KV Cache** | 重复计算 Key/Value 矩阵 | 缓存已计算的结果 | 做过的笔记不用重做 |
| **Flash Attention** | 注意力矩阵占内存太大 | 分块计算，减少内存读写 | 大仓库分区整理 |
| **GQA** | KV Cache 占显存太多 | 多个 Query 共享一组 KV | 多个窗口共用一个柜台 |

---


## 二、核心原理详解

### 2.1 KV Cache——以空间换时间

#### 问题：重复计算！

回忆 GPT 的自回归生成过程：


In [ ]:
第1步: 输入 [A]           → 计算 A 的 Q, K, V
第2步: 输入 [A, B]        → 重新计算 A 和 B 的 Q, K, V（A 被重复计算！）
第3步: 输入 [A, B, C]     → 重新计算 A, B, C 的 Q, K, V（A, B 被重复计算！）
...


每生成一个新 token，之前所有 token 的 K 和 V 都要重新计算——这是巨大的浪费！

#### 解决方案：缓存

既然之前的 K 和 V 不变，为什么不算一次存起来？


In [ ]:
第1步: 计算 Q₁, K₁, V₁                    → 缓存 [K₁, V₁]
第2步: 只算 Q₂, K₂, V₂                    → 缓存 [K₁,V₁, K₂,V₂]
       Q₂ × [K₁,K₂]ᵀ → 注意力权重 → × [V₁,V₂]
第3步: 只算 Q₃, K₃, V₃                    → 缓存 [K₁,V₁, K₂,V₂, K₃,V₃]
       Q₃ × [K₁,K₂,K₃]ᵀ → 注意力权重 → × [V₁,V₂,V₃]


**打个比方**：就像你做读书笔记——已经读过的页不需要重读，只需要读新的一页，把笔记加上去。

#### 性能提升

- 计算量：从 O(n²) 降低到 O(n)（每步只需处理新 token）
- 推理速度：提升 2-5 倍（特别是长文本生成）
- 代价：额外显存占用

#### 显存占用计算

KV Cache 的显存 = 2 × n_layers × seq_len × d_model × dtype_size

以 7B 模型（n_layers=32, d_model=4096）为例：
- 序列长度 2048，float16 → 约 1 GB
- 序列长度 8192，float16 → 约 4 GB
- 序列长度 32768，float16 → 约 16 GB

**结论**：KV Cache 的显存占用随序列长度线性增长，长文本生成时可能比模型本身还大！

### 2.2 Flash Attention——减少内存读写

#### 问题：标准注意力的内存瓶颈

标准注意力计算：


In [ ]:
# 输入: Q, K, V shape = (n, d)
scores = Q @ K.T          # (n, n) — 这个矩阵巨大！
attn = softmax(scores)    # (n, n)
output = attn @ V         # (n, d)


n=4096 时，`scores` 矩阵有 4096×4096 = 1600 万个元素——占用大量内存。

**核心问题**：中间矩阵 `scores` 和 `attn` 被写入 GPU HBM（高带宽内存），然后又被读出来进行下一步计算。这个读写操作非常慢。

#### Flash Attention 的解法

Flash Attention 的核心思想：**分块计算（Tiling），永不实例化完整的 n×n 矩阵**


In [ ]:
传统方式:
1. 计算 Q×K^T → 写入 HBM (n×n 矩阵)
2. 从 HBM 读取 → softmax → 写入 HBM (n×n 矩阵)
3. 从 HBM 读取 → ×V → 写入 HBM (n×d 矩阵)

Flash Attention:
在 GPU SRAM（快速缓存）中分块完成所有操作
永不写出 n×n 矩阵到 HBM


**打个比方**：
- 传统方式：你要组装一辆汽车，零件从远方仓库一件一件运过来，每次往返都花很多时间
- Flash Attention：你把零件分成几批，每批一次性运到工位旁边，在工位上完成组装

#### Flash Attention 的效果

- 速度：长序列注意力计算加速 2-4 倍
- 内存：从 O(n²) 降到 O(n)（不再需要 n×n 中间矩阵）
- 精度：数学上等价于标准注意力（不是近似！）

现代大模型（LLaMA-2/3、Qwen2、Mistral 等）默认使用 Flash Attention。

### 2.3 GQA（Grouped-Query Attention）——平衡效率与质量

#### 背景：MHA 的显存问题

标准注意力使用 MHA（Multi-Head Attention）：
- Q 有 n_heads 个头
- K 有 n_heads 个头
- V 有 n_heads 个头
- KV Cache 需要存储所有头的 K 和 V → 显存占用大

#### 演进：MHA → MQA → GQA

| 方案 | Q 头数 | K/V 头数 | KV Cache 大小 | 效果 | 代表模型 |
|------|--------|----------|-------------|------|---------|
| **MHA** | 32 | 32 | 100% | 最好 | GPT-2, BERT |
| **MQA** | 32 | 1 | 1/32 | 略差 | PaLM |
| **GQA** | 32 | 8 (每组4个Q共享) | 1/4 | 接近MHA | LLaMA-2/3, Qwen2 |

**打个比方**：
- MHA：32 个窗口各配一个专属办事员（效率高但人多）
- MQA：32 个窗口共用 1 个办事员（省钱但排队久）
- GQA：32 个窗口分成 8 组，每组共用 1 个办事员（平衡！）

#### 为什么 GQA 更受欢迎？

GQA 在效果和效率之间找到了最佳平衡点：
- 效果只比 MHA 差一点点（通常 <1% benchmark 差距）
- KV Cache 减少 75%（从 32 组 KV 降到 8 组）
- 推理速度提升 20-30%
- 显存节省意味着可以处理更长的上下文

---


## 三、代码实战

### 3.1 模拟 KV Cache 效果


In [ ]:
import time
import numpy as np

def generate_without_cache(seq_len, d_model=512):
    """无 KV Cache 的生成"""
    total_compute = 0
    for step in range(1, seq_len + 1):
        # 每步都要重新计算所有 token 的 K, V
        Q = np.random.randn(1, d_model)
        K = np.random.randn(step, d_model)  # 所有历史 token
        V = np.random.randn(step, d_model)
        
        scores = np.dot(Q, K.T) / np.sqrt(d_model)
        output = np.dot(scores, V)
        total_compute += step  # 累计计算量
    
    return total_compute

def generate_with_cache(seq_len, d_model=512):
    """有 KV Cache 的生成"""
    total_compute = 0
    for step in range(1, seq_len + 1):
        # 只计算新 token 的 Q, K, V
        Q = np.random.randn(1, d_model)
        K_new = np.random.randn(1, d_model)  # 只算新的
        V_new = np.random.randn(1, d_model)
        
        # K, V 从缓存读取，不重新计算
        total_compute += 1  # 每步只有 1 个 token 的计算量
    
    return total_compute

# 对比
for seq_len in [100, 500, 1000]:
    without = generate_without_cache(seq_len)
    with_c = generate_with_cache(seq_len)
    ratio = without / with_c
    print(f"序列长度 {seq_len:4d}: 无缓存={without:6d} 次 | 有缓存={with_c:4d} 次 | 加速比={ratio:.0f}x")


### 3.2 计算 KV Cache 显存


In [ ]:
def calculate_kv_cache_memory(model_name, n_layers, d_model, max_seq_len, 
                               n_kv_heads, head_dim, dtype_bytes=2):
    """计算 KV Cache 的显存占用"""
    # KV Cache 大小 = 2(K&V) × n_layers × max_seq_len × n_kv_heads × head_dim × dtype_bytes
    cache_size = (2 * n_layers * max_seq_len * n_kv_heads * head_dim * dtype_bytes)
    
    print(f"\n{model_name}:")
    print(f"  层数: {n_layers}, d_model: {d_model}")
    print(f"  KV 头数: {n_kv_heads}, 最大序列: {max_seq_len}")
    print(f"  KV Cache 大小: {cache_size / 1e9:.2f} GB ({cache_size / 1e6:.0f} MB)")
    return cache_size

# 主流模型对比
calculate_kv_cache_memory("LLaMA-7B (MHA, 32 heads)", 
                          n_layers=32, d_model=4096, max_seq_len=2048,
                          n_kv_heads=32, head_dim=128)

calculate_kv_cache_memory("LLaMA-2-7B (GQA, 8 KV heads)", 
                          n_layers=32, d_model=4096, max_seq_len=4096,
                          n_kv_heads=8, head_dim=128)

calculate_kv_cache_memory("Qwen2-7B (GQA, 4 KV heads)", 
                          n_layers=28, d_model=3584, max_seq_len=32768,
                          n_kv_heads=4, head_dim=128)


### 3.3 模拟 Flash Attention 的分块计算


In [ ]:
def standard_attention(Q, K, V):
    """标准注意力计算"""
    d_k = Q.shape[-1]
    scores = np.dot(Q, K.T) / np.sqrt(d_k)     # (n, n) 矩阵
    attn = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attn /= np.sum(attn, axis=-1, keepdims=True)
    return np.dot(attn, V)

def flash_attention_simplified(Q, K, V, block_size=64):
    """简化版 Flash Attention（分块计算）"""
    n = Q.shape[0]
    d_k = Q.shape[-1]
    output = np.zeros_like(Q)
    
    for i in range(0, n, block_size):
        Q_block = Q[i:i+block_size]  # 取一块 Q
        
        # 累积 softmax 的分母
        block_output = np.zeros_like(Q_block)
        block_max = np.full(Q_block.shape[0], -np.inf)
        block_sum = np.zeros(Q_block.shape[0])
        
        for j in range(0, n, block_size):
            K_block = K[j:j+block_size]
            V_block = V[j:j+block_size]
            
            # 计算 block 内的注意力
            scores = np.dot(Q_block, K_block.T) / np.sqrt(d_k)
            block_max_new = np.maximum(block_max, np.max(scores, axis=-1))
            
            # 更新累积值（这是 Flash Attention 的核心数学技巧）
            exp_diff = np.exp(block_max - block_max_new)
            exp_scores = np.exp(scores - block_max_new[:, None])
            
            block_sum = block_sum * exp_diff + np.sum(exp_scores, axis=-1)
            block_output = block_output * exp_diff[:, None] + np.dot(exp_scores, V_block)
            block_max = block_max_new
        
        # 最终归一化
        output[i:i+block_size] = block_output / block_sum[:, None]
    
    return output

# 验证两种方法结果一致
np.random.seed(42)
n, d = 256, 64
Q, K, V = np.random.randn(n, d), np.random.randn(n, d), np.random.randn(n, d)

result_standard = standard_attention(Q, K, V)
result_flash = flash_attention_simplified(Q, K, V, block_size=32)

max_diff = np.max(np.abs(result_standard - result_flash))
print(f"标准注意力 vs Flash Attention 最大差异: {max_diff:.10f}")
print("（差异极小说明两种方法数学上等价）")


---


## 四、可视化理解


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 图1: KV Cache 的加速效果
seq_lens = [50, 100, 200, 500, 1000, 2000]
without_cache = [n*(n+1)/2 for n in seq_lens]  # O(n²) 总计算量
with_cache = [n for n in seq_lens]               # O(n) 总计算量

axes[0].plot(seq_lens, without_cache, 'r-o', label='无 KV Cache (O(n²))', linewidth=2)
axes[0].plot(seq_lens, with_cache, 'b-s', label='有 KV Cache (O(n))', linewidth=2)
axes[0].fill_between(seq_lens, with_cache, without_cache, alpha=0.2, color='red')
axes[0].set_title('KV Cache: 总计算量对比', fontsize=13, fontweight='bold')
axes[0].set_xlabel('序列长度')
axes[0].set_ylabel('总计算量（相对值）')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 图2: Flash Attention 内存对比
seq_lens2 = np.array([512, 1024, 2048, 4096, 8192, 16384])
standard_mem = (seq_lens2 ** 2) * 4 / 1e9  # float32, n×n 矩阵
flash_mem = seq_lens2 * 64 * 4 / 1e9        # 只存 block 大小的中间结果

axes[1].bar(range(len(seq_lens2)), standard_mem, alpha=0.7, label='标准注意力', color='#e74c3c')
axes[1].bar(range(len(seq_lens2)), flash_mem, alpha=0.7, label='Flash Attention', color='#3498db')
axes[1].set_xticks(range(len(seq_lens2)))
axes[1].set_xticklabels([str(s) for s in seq_lens2])
axes[1].set_title('Flash Attention: 内存占用对比', fontsize=13, fontweight='bold')
axes[1].set_xlabel('序列长度')
axes[1].set_ylabel('中间结果内存 (GB)')
axes[1].legend()
axes[1].set_yscale('log')

# 图3: MHA vs MQA vs GQA
configs = ['MHA\n(32 KV heads)', 'GQA-8\n(8 KV heads)', 'GQA-4\n(4 KV heads)', 'MQA\n(1 KV head)']
kv_sizes = [32, 8, 4, 1]
quality_scores = [100, 98, 96, 93]  # 相对质量评分
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']

ax3a = axes[2]
ax3b = ax3a.twinx()
bars1 = ax3a.bar(configs, kv_sizes, alpha=0.7, color=colors, label='KV Cache 相对大小')
line1 = ax3b.plot(configs, quality_scores, 'r-o', linewidth=2, markersize=8, label='相对质量评分')
ax3a.set_title('MHA vs GQA vs MQA', fontsize=13, fontweight='bold')
ax3a.set_ylabel('KV Cache 相对大小', color='gray')
ax3b.set_ylabel('相对质量评分', color='red')
ax3b.set_ylim(85, 105)

lines1, labels1 = ax3a.get_legend_handles_labels()
lines2, labels2 = ax3b.get_legend_handles_labels()
ax3a.legend(lines1 + lines2, labels1 + labels2, loc='center left')

plt.tight_layout()
plt.savefig('/tmp/inference_optimization.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ 可视化图表已生成！")


---


## 五、业务关联

### 与 LangChat / Agent / 企业 AI 的关系

1. **并发处理**：KV Cache 让 LangChat 能同时服务多个用户——每个用户有自己的 KV Cache，互不干扰。理解 KV Cache 大小帮你估算单机能服务多少并发用户。

2. **长文本处理**：GQA + Flash Attention 让模型能处理 32K 甚至 128K 上下文。企业文档分析、代码生成等场景直接受益。

3. **成本控制**：理解推理优化的原理，你能准确估算部署成本——需要多大显存？几张卡？能服务多少 QPS？

4. **糖水店场景**：如果用 LangChat 搭建糖水店智能客服：
   - KV Cache → 每个对话会话占用一定显存，10 个并发用户就需要 10 份 KV Cache
   - Flash Attention → 让响应时间从 5 秒降到 1 秒，用户体验大幅提升
   - GQA → 选择 Qwen2（GQA-4）而不是 LLaMA-1（MHA），同样的 GPU 能服务更多用户

---


## 六、常见误区

### 误区 1: "KV Cache 是万能的"
**纠正**：KV Cache 只对**自回归生成**（GPT 模式）有效。BERT 式的一次性编码不需要 KV Cache（因为没有"逐步生成"的过程）。而且 KV Cache 本身会占用大量显存——序列越长，缓存越大。

### 误区 2: "Flash Attention 是注意力的近似"
**纠正**：Flash Attention 在数学上与标准注意力**完全等价**！它只是通过巧妙的分块计算和内存管理来加速，结果完全一样（浮点精度范围内）。

### 误区 3: "MQA 和 GQA 差不多"
**纠正**：虽然都是减少 KV 头数，但 MQA 过于激进（只留 1 个 KV 头），效果下降明显。GQA 找到了更好的平衡点——8 个 KV 头通常能达到接近 MHA 的效果。

---


## 🧪 课堂练习（5分钟）

1. **计算题**：一个 32 层、d_model=4096 的模型，float16 精度，处理 4096 长度的序列，KV Cache 需要多大的显存？

2. **概念题**：为什么 KV Cache 不能用于 BERT 式的 Encoder 模型？

3. **选择题**：Flash Attention 相比标准注意力的主要优势是？
   - A) 减少计算量
   - B) 减少内存读写次数
   - C) 使用了近似算法
   - D) 减少了注意力头数

---


## 📝 课后测试（15分钟）

1. **简答题**：解释 KV Cache 的工作原理。在生成第 N 个 token 时，它如何避免重复计算？

2. **计算题**：对比 MHA（32 KV heads）和 GQA-8（8 KV heads）在序列长度 8192、32 层、head_dim=128、float16 时的 KV Cache 显存差异。

3. **分析题**：Flash Attention 是如何在不实例化完整 n×n 注意力矩阵的情况下实现 softmax 的？解释 online softmax 的数学原理。

4. **设计题**：你需要在单张 RTX 4090（24GB 显存）上部署一个 13B 模型，要求支持 4K 上下文和 10 个并发用户。你会选择什么优化策略？

5. **业务题**：LangChat 部署时，如果用户对话平均 20 轮、每轮 50 字（中文），计算单个用户的 KV Cache 显存占用（以 Qwen2-7B 为例）。

---


## 🔑 今日术语

| 英文 | 音标 | 中文解释 |
|------|------|---------|
| KV Cache | [keɪ-viː kæʃ] | 键值缓存，存储已计算的 Key/Value 避免重复计算 |
| Flash Attention | [flæʃ əˈtenʃən] | 闪电注意力，通过分块计算减少内存读写的优化技术 |
| GQA (Grouped-Query Attention) | [ɡruːpt ˈkwɪəri əˈtenʃən] | 分组查询注意力，多个 Q 共享一组 KV |
| MQA (Multi-Query Attention) | [ˈmʌlti ˈkwɪəri əˈtenʃən] | 多查询注意力，所有 Q 共享 1 组 KV（最激进） |
| MHA (Multi-Head Attention) | [ˈmʌlti hɛd əˈtenʃən] | 多头注意力，标准 Transformer 注意力机制 |
| HBM (High Bandwidth Memory) | [haɪ ˈbændwɪdθ ˈmɛməri] | 高带宽内存，GPU 的主内存 |
| SRAM | [ɛs-ræm] | 静态随机存储器，GPU 片上快速缓存 |
| Autoregressive | [ˌɔːtoʊrɪˈɡrɛsɪv] | 自回归，基于之前输出逐步生成新 token |
| Throughput | [ˈθruːpʊt] | 吞吐量，单位时间处理的请求数 |
| Latency | [ˈleɪtənsi] | 延迟，单个请求的响应时间 |

---


## 📎 参考资源

- 📄 [FlashAttention: Fast and Memory-Efficient Exact Attention (2022)](https://arxiv.org/abs/2205.14135) - Flash Attention 论文
- 📄 [GQA: Training Generalized Multi-Query Transformer Models (2023)](https://arxiv.org/abs/2305.13245) - GQA 论文
- 📄 [Fast Transformer Decoding: One-Write KV Cache (Google, 2019)](https://arxiv.org/abs/1911.02150) - KV Cache 原理
- 📄 [FlashAttention-2: Faster Attention with Better Parallelism (2023)](https://arxiv.org/abs/2307.08691) - Flash Attention v2
- 🎥 [GPU Memory Hierarchy - NVIDIA](https://docs.nvidia.com/cuda/cuda-c-programming-guide/) - GPU 内存层级
- 🔧 [vLLM: PagedAttention for efficient KV Cache management](https://arxiv.org/abs/2309.06180) - vLLM 推理框架
- 🔧 [The Illustrated Transformer - Jay Alammar](https://jalammar.github.io/illustrated-transformer/) - Transformer 可视化

---

> 💡 **明日预告**：Day 6 进入实战环节——QLoRA 微调！我们将学习如何用 4-bit 量化 + LoRA 适配器在消费级 GPU 上微调大模型，打造你的专属糖水店 AI 助手！
